In [1]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from requests.exceptions import ReadTimeout
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.endpoints import playergamelogs
from nba_api.stats.endpoints import scoreboardv2
from nba_api.stats.static import teams
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

time.sleep(1)

# Function to fetch the roster for a given team
def get_team_roster(team_id):
    roster = commonteamroster.CommonTeamRoster(team_id=team_id)  # Instantiate the endpoint
    roster_df = roster.get_data_frames()[0]  # Extract the DataFrame
    return roster_df[['PLAYER_ID', 'PLAYER']]

from nba_api.stats.endpoints import playergamelog

time.sleep(1)

# Function to calculate stats against a given team
def calculate_stats_against_team(player_id, opponent_team_abbr, season='2024', season_type='Regular Season'):
    retries = 3  # Number of retries
    for attempt in range(retries):
        try:
            gamelog = playergamelog.PlayerGameLog(
                player_id=player_id,
                season=season,
                season_type_all_star=season_type,
                timeout=60
            ).get_data_frames()[0]  # Extract the first DataFrame
            break
        except ReadTimeout:
            print(f"Timeout occurred. Retrying {attempt + 1}/{retries}...")
            time.sleep(5)  # Wait 5 seconds before retrying
    else:
        raise ReadTimeout("Failed to fetch data after multiple attempts.")

    player_logs = gamelog
    games_against_team = player_logs[player_logs['MATCHUP'].str.contains(opponent_team_abbr, na=False)]

    total_points = games_against_team['PTS'].sum()
    total_rebounds = games_against_team['REB'].sum()
    total_assists = games_against_team['AST'].sum()

    games_played = len(games_against_team)
    ppg = total_points / games_played if games_played > 0 else 0
    rpg = total_rebounds / games_played if games_played > 0 else 0
    apg = total_assists / games_played if games_played > 0 else 0

    return ppg, rpg, apg

time.sleep(1)

# Function to fetch today's matchups and analyze stats
def get_ppg_rpg_apg_for_todays_games():
    matchups = fetch_schedule()  # Correct function
    results = []

    for game in matchups:
        home_team_id = game['home_team_id']
        visitor_team_id = game['away_team_id']

        home_team_roster = get_team_roster(home_team_id)
        visitor_team_roster = get_team_roster(visitor_team_id)

        home_team_abbr = game['home_team']
        visitor_team_abbr = game['away_team']

        for _, player in home_team_roster.iterrows():
            ppg, rpg, apg = calculate_stats_against_team(player['PLAYER_ID'], visitor_team_abbr)
            results.append({
                'Player': player['PLAYER'],
                'Team': home_team_abbr,
                'Opponent': visitor_team_abbr,
                'PPG_Against_Opponent': ppg,
                'RPG_Against_Opponent': rpg,
                'APG_Against_Opponent': apg
            })

        for _, player in visitor_team_roster.iterrows():
            ppg, rpg, apg = calculate_stats_against_team(player['PLAYER_ID'], home_team_abbr)
            results.append({
                'Player': player['PLAYER'],
                'Team': visitor_team_abbr,
                'Opponent': home_team_abbr,
                'PPG_Against_Opponent': ppg,
                'RPG_Against_Opponent': rpg,
                'APG_Against_Opponent': apg
            })

    df = pd.DataFrame(results)
    display(df)
    return df

time.sleep(1)

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

time.sleep(1)

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

time.sleep(1)

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

time.sleep(1)

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

time.sleep(10)

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

time.sleep(1)

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
# Main analysis function
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    # Ensure games_today has been fetched
    games_today = fetch_schedule()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []  # You need to define this to collect player data
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    # Process player data as needed
                    # Your processing logic goes here

            for game in games_today:
                home_team = game['home_team']
                away_team = game['away_team']

                for team_name in [home_team, away_team]:
                    team_id = teams.find_team_by_abbreviation(team_name)['id']
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                    for _, player in roster.iterrows():
                        player_position = player['POSITION']
                        # Continue processing

                    if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                    for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                ppg_diff = mean_ppg - std_dev_ppg
                                rpg_diff = mean_rpg - std_dev_rpg
                                apg_diff = mean_apg - std_dev_apg

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': round(ppg_diff, 2),
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': round(rpg_diff, 2),
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': round(apg_diff, 2),
                                    'Adj APG': round(adjusted_apg, 2),
                                    'Usage': usg_pct,
                                    'Injury Status': injury_status,
                                    'Position': player_position
                                })
                        except Exception as e:
                            print(f"Error processing data for {player_name}: {e}")
                            continue

                            time.sleep(1)  # To avoid rate-limiting

        all_players_df = pd.DataFrame(all_players_data)
        display(all_players_df)
        return all_players_df

# Fetch and analyze data for today
get_ppg_rpg_apg_for_todays_games()
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 4.5 MB/s eta 0:00:00
Timeout occurred. Retrying 1/3...
Timeout occurred. Retrying 2/3...
Timeout occurred. Retrying 1/3...
Timeout occurred. Retrying 1/3...
Timeout occurred. Retrying 2/3...
Timeout occurred. Retrying 3/3...


ReadTimeout: Failed to fetch data after multiple attempts.